In [8]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from numpy import pi

In [9]:
def build_cuccaro_circuit(a, b):
    qreg_q = QuantumRegister(10, 'q')
    creg_c = ClassicalRegister(5, 'c')
    circuit = QuantumCircuit(qreg_q, creg_c)

    # q1,q4,q7,q10 = bits of A (LSB to MSB)
    # q2,q5,q8,q11 = bits of B (LSB to MSB)

    # encode A
    if a & 1: circuit.x(qreg_q[1])
    if a & 2: circuit.x(qreg_q[4])
    if a & 4: circuit.x(qreg_q[6])
    if a & 8: circuit.x(qreg_q[8])

    # encode B
    if b & 1: circuit.x(qreg_q[0])
    if b & 2: circuit.x(qreg_q[3])
    if b & 4: circuit.x(qreg_q[5])
    if b & 8: circuit.x(qreg_q[7])

    circuit.barrier()

    circuit.cx(qreg_q[6], qreg_q[5])
    circuit.cx(qreg_q[4], qreg_q[3])
    circuit.cx(qreg_q[8], qreg_q[7])
    circuit.cx(qreg_q[4], qreg_q[2])
    circuit.ccx(qreg_q[0], qreg_q[1], qreg_q[2])
    circuit.cx(qreg_q[6], qreg_q[4])
    circuit.ccx(qreg_q[2], qreg_q[3], qreg_q[4])
    circuit.cx(qreg_q[8], qreg_q[6])
    circuit.ccx(qreg_q[4], qreg_q[5], qreg_q[6])
    circuit.cx(qreg_q[8], qreg_q[9])
    circuit.barrier(qreg_q[3])
    circuit.x(qreg_q[3])
    circuit.ccx(qreg_q[6], qreg_q[7], qreg_q[9])
    circuit.x(qreg_q[5])
    circuit.cx(qreg_q[2], qreg_q[3])
    circuit.cx(qreg_q[4], qreg_q[5])
    circuit.cx(qreg_q[6], qreg_q[7])
    circuit.ccx(qreg_q[4], qreg_q[5], qreg_q[6])
    circuit.ccx(qreg_q[2], qreg_q[3], qreg_q[4])
    circuit.x(qreg_q[5])
    circuit.cx(qreg_q[8], qreg_q[6])
    circuit.ccx(qreg_q[0], qreg_q[1], qreg_q[2])
    circuit.x(qreg_q[3])
    circuit.cx(qreg_q[6], qreg_q[4])
    circuit.barrier(qreg_q[8])
    circuit.cx(qreg_q[4], qreg_q[2])
    circuit.barrier(qreg_q[0])
    circuit.barrier(qreg_q[6])
    circuit.barrier(qreg_q[8])
    circuit.cx(qreg_q[6], qreg_q[5])
    circuit.cx(qreg_q[8], qreg_q[7])
    circuit.cx(qreg_q[1], qreg_q[0])
    circuit.cx(qreg_q[4], qreg_q[3])
    circuit.measure(qreg_q[0], creg_c[0])
    circuit.measure(qreg_q[3], creg_c[1])
    circuit.measure(qreg_q[5], creg_c[2])
    circuit.measure(qreg_q[7], creg_c[3])
    circuit.measure(qreg_q[9], creg_c[4])


    return circuit

In [10]:
import numpy as np

shots = 1000

# ====================================================
# BITFLIP NOISE MODEL
# ====================================================

p = 0.0347

# ----------------------------------------------------
# 1-Qubit Bitflip Error
# ----------------------------------------------------

error_1 = pauli_error([
    ('X', p),
    ('I', 1-p)
])

# ----------------------------------------------------
# 2-Qubit Error
# E ⊗ E
# ----------------------------------------------------

error_2 = error_1.tensor(error_1)

# ----------------------------------------------------
# 3-Qubit Error
# E ⊗ E ⊗ E
# ----------------------------------------------------

error_3 = error_1.tensor(error_1).tensor(error_1)

# ====================================================
# CREATE NOISE MODEL
# ====================================================

noise_model = NoiseModel()

# ----------------------------------------------------
# Apply noise to X gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_1,
    ['x']
)

# ----------------------------------------------------
# Apply noise to CX gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_2,
    ['cx']
)

# ----------------------------------------------------
# Apply noise to CCX gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_3,
    ['ccx']
)

sim = AerSimulator(noise_model=noise_model)

# ====================================================
# NUMPY ARRAYS FOR METRICS
# ====================================================

ER_arr = np.zeros((16, 16))
NMED_arr = np.zeros((16, 16))
MRED_arr = np.zeros((16, 16))

for a in range(16):
    for b in range(16):

        qc = build_cuccaro_circuit(a, b)

        compiled = transpile(qc, sim)

        counts = sim.run(
            compiled,
            shots=shots
        ).result().get_counts()

        correct = a + b
        correct_output = format(correct, '05b')

        correct_counts = counts.get(correct_output, 0)

        ER = 1 - correct_counts / shots

        total_ED = 0
        total_relative_ED = 0

        for output, freq in counts.items():

            noisy_decimal = int(output, 2)

            ED = abs(correct - noisy_decimal)

            total_ED += ED * freq

            if correct != 0:
                total_relative_ED += (ED / correct) * freq

        mean_ED = total_ED / shots

        D = 30

        NMED = mean_ED / D
        MRED = total_relative_ED / shots

        ER_arr[a, b] = ER
        NMED_arr[a, b] = NMED
        MRED_arr[a, b] = MRED

# ====================================================
# OVERALL METRICS
# ====================================================

er = np.mean(ER_arr)
nmed = np.mean(NMED_arr)
mred = np.mean(MRED_arr)

print("Bitflip Noise\n")

print(f"ER (%) : {er * 100:.2f}")
print(f"NMED   : {nmed:.3f}")
print(f"MRED   : {mred:.3f}")

# Flatten to 1D if needed
er_array = ER_arr.flatten()

print("\nER Array:")
print(er_array)

Bitflip Noise

ER (%) : 79.75
NMED   : 0.166
MRED   : 0.493

ER Array:
[0.87 0.8  0.72 0.79 0.78 0.8  0.74 0.68 0.79 0.86 0.69 0.83 0.83 0.75
 0.73 0.85 0.91 0.79 0.74 0.81 0.76 0.78 0.84 0.74 0.75 0.78 0.78 0.91
 0.76 0.83 0.81 0.75 0.74 0.8  0.79 0.82 0.8  0.72 0.87 0.75 0.79 0.87
 0.76 0.79 0.75 0.74 0.77 0.78 0.76 0.79 0.8  0.78 0.72 0.83 0.84 0.78
 0.77 0.77 0.83 0.79 0.82 0.82 0.83 0.79 0.83 0.78 0.84 0.79 0.84 0.77
 0.86 0.82 0.8  0.77 0.79 0.72 0.82 0.76 0.8  0.74 0.82 0.82 0.81 0.74
 0.75 0.78 0.77 0.84 0.71 0.75 0.8  0.83 0.78 0.84 0.79 0.84 0.77 0.8
 0.79 0.76 0.79 0.76 0.86 0.8  0.75 0.78 0.78 0.74 0.83 0.75 0.89 0.8
 0.74 0.79 0.77 0.8  0.8  0.83 0.86 0.8  0.67 0.78 0.77 0.87 0.78 0.78
 0.86 0.82 0.75 0.78 0.74 0.79 0.73 0.86 0.75 0.75 0.84 0.86 0.8  0.76
 0.77 0.83 0.74 0.75 0.82 0.85 0.8  0.83 0.79 0.81 0.77 0.8  0.85 0.84
 0.84 0.83 0.81 0.83 0.82 0.82 0.75 0.81 0.87 0.85 0.79 0.88 0.81 0.74
 0.89 0.82 0.82 0.78 0.84 0.79 0.79 0.77 0.72 0.78 0.73 0.81 0.78 0.74
 0.8  0.